In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import sys
from tqdm import tqdm


In [34]:
from btc_model.strategy.funding_rate_arbitrage.pair_selector import PairSelector
from btc_model.strategy.funding_rate_arbitrage.exchange_connector import ExchangeConnector
from btc_model.strategy.funding_rate_arbitrage.whitelist_manager import FundingRateWhitelistManager
from btc_model.core.util.crypto_util import CryptoUtil
from btc_model.core.backend.websocket_service import WebSocketService
from btc_model.core.wrapper.db_wrapper import DBWrapper


In [48]:

print("初始化交易所连接器...")
connector = ExchangeConnector('okx') # 确保 OKX API 密钥在 .env 中配置

if connector.exchange:
    print("初始化交易对筛选器...")
    pair_selector = PairSelector(connector)

    # 运行筛选
    filtered_pairs = pair_selector.get_filtered_pairs()

    print("--- 筛选结果 ---")
    if filtered_pairs:
        print("找到以下适用的交易对 (格式与 config.TARGET_PAIRS 一致):")
        import json
        print(json.dumps(filtered_pairs, indent=4))
    else:
        print("未找到适用的交易对。请检查配置、API密钥、网络连接以及交易所的交易对情况。")

    connector.close()
    print("程序退出。")
else:
    print("交易所连接失败，无法运行交易对筛选器示例。请检查 API 密钥和网络。")

INFO   : 2025-05-15 15:37:50,046 >>> 正在连接到交易所: okx
INFO   : 2025-05-15 15:37:50,082 >>> OKX 连接器设置为模拟盘模式


初始化交易所连接器...


INFO   : 2025-05-15 15:37:52,043 >>> 成功连接并加载市场信息: okx
INFO   : 2025-05-15 15:37:52,044 >>> 初始化交易对筛选器...
INFO   : 2025-05-15 15:37:52,095 >>> 开始筛选适用的交易对...
INFO   : 2025-05-15 15:37:52,097 >>> 获取到 646 个活跃现货市场，105 个活跃 USDT 永续合约市场。
INFO   : 2025-05-15 15:37:52,097 >>> 找到 94 个潜在的现货-永续合约配对。
INFO   : 2025-05-15 15:37:52,098 >>> 正在对潜在配对进行流动性筛选...


初始化交易对筛选器...


INFO   : 2025-05-15 15:37:52,549 >>> 识别到适用交易对: BTC-USDT-HEDGE (现货: BTC/USDT, 永续: BTC/USDT:USDT)
INFO   : 2025-05-15 15:37:52,550 >>> 识别到适用交易对: ETH-USDT-HEDGE (现货: ETH/USDT, 永续: ETH/USDT:USDT)
INFO   : 2025-05-15 15:37:52,550 >>> 识别到适用交易对: LTC-USDT-HEDGE (现货: LTC/USDT, 永续: LTC/USDT:USDT)
INFO   : 2025-05-15 15:37:52,551 >>> 识别到适用交易对: BCH-USDT-HEDGE (现货: BCH/USDT, 永续: BCH/USDT:USDT)
INFO   : 2025-05-15 15:37:52,552 >>> 识别到适用交易对: DOGE-USDT-HEDGE (现货: DOGE/USDT, 永续: DOGE/USDT:USDT)
INFO   : 2025-05-15 15:37:52,552 >>> 识别到适用交易对: SOL-USDT-HEDGE (现货: SOL/USDT, 永续: SOL/USDT:USDT)
INFO   : 2025-05-15 15:37:52,552 >>> 识别到适用交易对: SATS-USDT-HEDGE (现货: SATS/USDT, 永续: SATS/USDT:USDT)
INFO   : 2025-05-15 15:37:52,553 >>> 识别到适用交易对: 1INCH-USDT-HEDGE (现货: 1INCH/USDT, 永续: 1INCH/USDT:USDT)
INFO   : 2025-05-15 15:37:52,553 >>> 识别到适用交易对: AAVE-USDT-HEDGE (现货: AAVE/USDT, 永续: AAVE/USDT:USDT)
INFO   : 2025-05-15 15:37:52,554 >>> 识别到适用交易对: ADA-USDT-HEDGE (现货: ADA/USDT, 永续: ADA/USDT:USDT)
INFO   : 2025-05-15 15:37

--- 筛选结果 ---
找到以下适用的交易对 (格式与 config.TARGET_PAIRS 一致):
[
    {
        "pair_name": "BTC-USDT-HEDGE",
        "spot_symbol": "BTC/USDT",
        "perpetual_symbol": "BTC/USDT:USDT",
        "base_currency": "BTC",
        "quote_currency": "USDT"
    },
    {
        "pair_name": "ETH-USDT-HEDGE",
        "spot_symbol": "ETH/USDT",
        "perpetual_symbol": "ETH/USDT:USDT",
        "base_currency": "ETH",
        "quote_currency": "USDT"
    },
    {
        "pair_name": "LTC-USDT-HEDGE",
        "spot_symbol": "LTC/USDT",
        "perpetual_symbol": "LTC/USDT:USDT",
        "base_currency": "LTC",
        "quote_currency": "USDT"
    },
    {
        "pair_name": "BCH-USDT-HEDGE",
        "spot_symbol": "BCH/USDT",
        "perpetual_symbol": "BCH/USDT:USDT",
        "base_currency": "BCH",
        "quote_currency": "USDT"
    },
    {
        "pair_name": "DOGE-USDT-HEDGE",
        "spot_symbol": "DOGE/USDT",
        "perpetual_symbol": "DOGE/USDT:USDT",
        "base_currency": "DO

In [50]:
funding_rate_arbitrage_whitelist = []
for pair in tqdm(filtered_pairs):
    pair_name = pair['pair_name']
    perpetual_symbol = pair['perpetual_symbol']
    funding_rate_arbitrage_whitelist.append({'symbol': pair_name} | CryptoUtil.analyze_funding_history(connector.exchange, perpetual_symbol))
    
print(funding_rate_arbitrage_whitelist)

100%|██████████| 77/77 [00:12<00:00,  5.93it/s]

[{'symbol': 'BTC-USDT-HEDGE', 'current_rate': 3.32955710301e-05, 'avg_rate': 0.00017806256107282334, 'max_rate': 0.00375, 'min_rate': -0.0003262903294582, 'volatility': 0.0006686424655859445, 'trend': 'stable', 'risk_score': 0.19145037077636065}, {'symbol': 'ETH-USDT-HEDGE', 'current_rate': -5.62918775643e-05, 'avg_rate': 3.041442200220006e-06, 'max_rate': 0.0015900310533338, 'min_rate': -0.0016418266108859, 'volatility': 0.0004460371718406537, 'trend': 'up', 'risk_score': 0.32901390541301123}, {'symbol': 'LTC-USDT-HEDGE', 'current_rate': 0.0075, 'avg_rate': -0.0005123692543424399, 'max_rate': 0.0075, 'min_rate': -0.0075, 'volatility': 0.004719893403295858, 'trend': 'up', 'risk_score': 0.8831936041977515}, {'symbol': 'BCH-USDT-HEDGE', 'current_rate': -9.91560064287e-05, 'avg_rate': -0.0001577144501690133, 'max_rate': 0.0005131509821614, 'min_rate': -0.0009682960757082, 'volatility': 0.0002476053347713065, 'trend': 'stable', 'risk_score': 0.16882256034342638}, {'symbol': 'DOGE-USDT-HEDG

In [21]:
CryptoUtil.analyze_funding_history(connector.exchange, 'BTC/USDT:USDT')

{'current_rate': 8.95267539494e-05,
 'avg_rate': 0.00016037239839539666,
 'max_rate': 0.00375,
 'min_rate': -0.0003262903294582,
 'volatility': 0.000673154089789614,
 'trend': 'up',
 'risk_score': 0.34397031554535284}

In [43]:
db_wrapper = DBWrapper().get_instance()
whitelist_manager = FundingRateWhitelistManager(db_wrapper, None)

whitelist_manager.load_whitelist()
white_list = whitelist_manager.get_whitelist()


INFO   : 2025-05-15 10:03:24,521 >>> Loaded 77 trading pairs from whitelist
INFO   : 2025-05-15 10:03:24,522 >>> FundingRateWhitelistManager initialized with 77 trading pairs
INFO   : 2025-05-15 10:03:24,558 >>> Loaded 77 trading pairs from whitelist


In [44]:
white_list

[{'exchange_id': 'okx',
  'base_currency': '1INCH',
  'quote_currency': 'USDT',
  'spot_inst_id': '1INCH/USDT',
  'swap_inst_id': '1INCH/USDT:USDT',
  'avg_ann_funding_rate_90d': None,
  'median_ann_funding_rate_90d': None,
  'std_dev_ann_funding_rate_90d': None,
  'positive_rate_pct_90d': None,
  'avg_basis_90d': None,
  'median_basis_90d': None,
  'std_dev_basis_90d': None,
  'max_basis_90d': None,
  'min_basis_90d': None,
  'basis_quartile_1_90d': None,
  'basis_quartile_3_90d': None,
  'basis_volatility_90d': None,
  'positive_basis_pct_90d': None,
  'basis_funding_correlation_90d': None,
  'avg_basis_to_funding_ratio_90d': None,
  'potential_apr_90d': None,
  'sharpe_ratio_90d': None,
  'max_drawdown_90d': None,
  'is_active': 1,
  'comment': None,
  'created_at': datetime.datetime(2025, 5, 15, 9, 53, 4),
  'updated_at': datetime.datetime(2025, 5, 15, 9, 53, 4)},
 {'exchange_id': 'okx',
  'base_currency': 'AAVE',
  'quote_currency': 'USDT',
  'spot_inst_id': 'AAVE/USDT',
  'swap_i

In [51]:
for pair in tqdm(filtered_pairs):
    whitelist_manager.add_to_whitelist(
        exchange_id=connector.exchange.id,
        base_currency=pair['base_currency'],
        quote_currency=pair['quote_currency'],
        spot_inst_id=pair['spot_symbol'],
        swap_inst_id=pair['perpetual_symbol'],
    )

  0%|          | 0/77 [00:00<?, ?it/s]ERROR  : 2025-05-15 15:39:06,995 >>> Error adding to whitelist: (pymysql.err.OperationalError) (2006, "MySQL server has gone away (BrokenPipeError(32, 'Broken pipe'))")
[SQL: 
            INSERT INTO funding_rate_arbitrage_whitelist (
                exchange_id, 
                base_currency, 
                quote_currency, 
                spot_inst_id, 
                swap_inst_id, 
                comment, 
                is_active
            ) VALUES (
                %(exchange_id)s, 
                %(base_currency)s, 
                %(quote_currency)s, 
                %(spot_inst_id)s, 
                %(swap_inst_id)s, 
                %(comment)s, 
                %(is_active)s
            )on duplicate key update 
                exchange_id = %(exchange_id)s,
                base_currency = %(base_currency)s,
                quote_currency = %(quote_currency)s,
                spot_inst_id = %(spot_inst_id)s,
                swap